## Step1 : Import the relevant libraries

In [4]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split


## Step 2: Load the dataset

In [5]:
cleaned_df = pd.read_csv("../data/processed/cleaned_data.csv")

## step 3: Split the features and the target, then split into train and test sets
- Split the target from the features
- X => is a 2D array

In [6]:
X = cleaned_df.drop(columns=['status_group'])
y = cleaned_df['status_group']

In [7]:
X = cleaned_df.drop(columns=['date_recorded', 'construction_year'])

In [8]:
X.shape

(52507, 16)

### Split into train and test

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y,
                                                    test_size=0.3,
                                                    stratify=y, # Stratify the target to maitain class distribution in train and test sets. our dataset is highly imbalanced
                                                    random_state=42)

In [10]:
((cleaned_df['status_group'].value_counts())/len(cleaned_df)) * 100

status_group
functional                 54.722228
non functional             38.048260
functional needs repair     7.229512
Name: count, dtype: float64

train - (54:38:7)  test - (54:38:7) - stratified helps maintain the ratios
-if you dont have stratified the proportions might be displayed this way
-Train (60, 40, 0)
- Test (50, 25, 25)

## step 4:

## Encode the target column
__Common types of encoders__
- Ordinal encoder: A type of encoding where there is a specific natural ranking or order

|Eductaion Level| encoded education level |
|---------------|-------------------------|
|Primary        |0                        |
|High School    |1                        |
|Bachelors      |2                        |
|Masters        |3                        |
|PHD            |4                        |

- Label encoding: where categories are labelled using integers

|Cities     | cities encoded |
|-----------|----------------|
|Nairobi    |0               |
|Mombasa    |1               |
|Kisumu     |2               |

_Limitation_ - The model might assume that there is a ranking to it.
eg can assume kis < Momb < Nai
_Best fit for_ - Target column


- Onehot Encoding - gives the columns a new name  

|Cities |
|-------|
|Nairobi|
|Mombasa|
|Kisumu | 
|Nairobi|      

_After onehot encoding_     
|cities_Nairobi|cities_Mombasa|cities_Kisumu|
|--------------|--------------|-------------|
|1             |0             |0            |
|0             |1             |0            |
|0             |0             |1            |
|1             |0             |0            |

_Limitation_ - it adds the dimensionality of the dataset hence use it sparingly.


In [12]:
#Encode the target column
le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.fit(y_test)


## Step 4 : Building the preprocessing pipeline.
- Why use sklearn pipeline?
- Sklearn pipelines bundle data preprocessing and model training into a single, cohesive object, making your machine learning workflows cleaner, safer and production ready.
- Instead of manually tracking and applying multiple `.fit()` and `.transform()` methosd across your training and testing sets, a pipeline allows you to execute the entire sequence with a single command.

In [19]:
# Numerical columns

numerical_cols = X_train.select_dtypes(include=['int', 'float']).columns.tolist()

numerical_cols.remove('month_recorded')
print(numerical_cols)



# Categorical columns
categorical_cols = X_train.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

categorical_cols.append('month_recorded')
print(categorical_cols)

['amount_tsh', 'gps_height', 'population', 'age']
['basin', 'region', 'scheme_management', 'permit', 'extraction_type_class', 'payment_type', 'quality_group', 'quantity_group', 'source_type', 'waterpoint_type_group', 'status_group', 'month_recorded']


In [20]:
### 5.1 Numerical pipeline
num_pipeline = Pipeline([('scaler', StandardScaler())])


# You can fill nulls using this pipeline.
# num_pipeline = Pipeline([('imputer', SimpleImputer(strategy = 'median'))('scaler', StandardScaler())])

In [23]:
### 5.2 Categorical pipeline

cat_pipeline = Pipeline([('encoder',
                          OneHotEncoder(handle_unknown='ignore'))])

# the handle_unkown='ignore helps avoid errors during transformation if the test set has unseen categorical

### Combine the numerical and categorical pipelines using columntransformer

In [24]:
preprocessor = ColumnTransformer(transformers=
                                 [('num', num_pipeline, numerical_cols),
                                  'cat', cat_pipeline, categorical_cols])
